In [2]:
import polars as pl
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
DATASET_PATH = "/group/pmc021/amunif/epi-thesis/workflow/14_E066 with alignment dataset/dataset"

In [4]:
def load_histone(filename):
    # Create a schema
    intersect_schema = pl.Schema({
        # The E066.bed files
        'chromosome_name': pl.String,
        'start': pl.Int64,
        'end': pl.Int64,
        'gene_id': pl.String,
        'E066': pl.Float64,
        'strand': pl.String,
        'label': pl.String,
        'external_gene_name': pl.String,
        'start_position': pl.Int64,
        'end_position': pl.Int64,
        'tss': pl.Int64,

        # The gappedPeak column
        "chrom_p": pl.String,
        "chromStart_p": pl.Int64, 
        "chromEnd_p": pl.Int64, 
        "name_p": pl.String, 
        "score_p": pl.Float64, 
        "strand_p": pl.String,
        "thickStart": pl.Int64,
        "thickEnd": pl.Int64,
        "itemRgb": pl.Int64,
        "blockCount": pl.Int64,
        "blockSizes": pl.String,
        "blockStarts": pl.String,
        "signalValue": pl.Float64,
        "pValue": pl.Float64,
        "qValue": pl.Float64
    })

    # Open file
    histone_df = pl.read_csv(
            filename,
            separator="\t",
            has_header = False,
            schema = intersect_schema   
        )
    
    return histone_df

In [5]:
def load_gene_expression(filename):
    # Create a schema
    E066_schema = pl.Schema({
        'chromosome_name': pl.String,
        'start': pl.Int64,
        'end': pl.Int64,
        'gene_id': pl.String,
        'E066': pl.Float64,
        'strand': pl.Int64,
        'label': pl.String,
        'external_gene_name': pl.String,
        'start_position': pl.Int64,
        'end_position': pl.Int64,
        'tss': pl.Int64
    })

    # Read the file
    df = pl.read_csv(filename, has_header=False, schema=E066_schema, separator="\t")

    return df
    

In [6]:
def create_empty_dataframe(histone_name):
    schema = pl.Schema({
        'gene_id': pl.String,
        histone_name: pl.List(pl.Float64),
        f'{histone_name}_wc': pl.UInt32,
        f'{histone_name}_len': pl.UInt32
    })

    df = pl.DataFrame(schema=schema)

    return df

In [7]:
def build_matrix(genes_df, histone_df, histone_name):
    # Build the dataframe with window
    genes_with_windows = genes_df.with_columns([
        pl.int_ranges(pl.col('start'), pl.col('end'), 100).alias('window_start')
    ]).explode('window_start')

    genes_with_windows = genes_with_windows.with_columns([
            (pl.col('window_start') + 100).alias('window_end')
    ])

    # Join genes with histone data
    joined_df = genes_with_windows.join(
        histone_df,
        left_on='gene_id',
        right_on='gene_id',
        how='left'
    )

    # Filter and calculate average signal value
    result_df = joined_df.filter(
        (pl.col('chromStart_p') < pl.col('window_end')) &
        (pl.col('chromEnd_p') > pl.col('window_start'))
    ).group_by(['gene_id', 'window_start'], maintain_order=True).agg([
        pl.col('signalValue').mean().alias(histone_name)
    ]).sort(['gene_id', 'window_start'])

    # Find the gene without histone match
    genes_wo_histone = genes_with_windows.join(
        result_df,
        on=["gene_id", "window_start"],
        how="anti"
    )

    # Add the signalValue column so it can be merged
    genes_wo_histone = genes_wo_histone.with_columns(
        signalValue = pl.lit(0.0).cast(pl.Float64)
    )

    # Aggregate the genes without histone result
    genes_wo_histone = genes_wo_histone.group_by(['gene_id', 'window_start'], maintain_order=True).agg([
            pl.col('signalValue').mean().alias(histone_name)
        ]).sort(['gene_id', 'window_start'])

    # Merge both (results and genes without histone)
    result_df.extend(genes_wo_histone)

    # Sort the result dataframe by gene_id and window start for aggregation
    sorted_result_df = result_df.sort(['gene_id', 'window_start'])
    
    # Group by to make array of features
    matrix_df = (
        sorted_result_df
        .with_columns(pl.col(histone_name).fill_null(0))
        .group_by(['gene_id'], maintain_order=True)
        .agg(pl.col(histone_name))
        .sort('gene_id')
    )

    # Add the count and length of array feature for checking
    matrix_control_df = matrix_df.with_columns(
        pl.col(histone_name)
        .list.eval(pl.element().is_not_null() & (pl.element() > 0))
        .list.sum()
        .alias(f"{histone_name}_wc")
    )

    matrix_control_df = matrix_control_df.with_columns(
        pl.col(histone_name).list.len().alias(f"{histone_name}_len")
    )

    # Finally, return the gene_id with its histone features
    return matrix_control_df

In [8]:
# Getting histone in chunk
def get_histone_features(genes_df, histone_df, histone_name):
    
    genes_w_histone = create_empty_dataframe(histone_name)
    
    for i, chunk in enumerate(genes_df.iter_slices(n_rows=100)):
        if i % 100 == 0:
            print(f"Processing {histone_name}: {i*100}/{genes_df.height}")
        
        result = build_matrix(chunk, histone_df, histone_name)
        genes_w_histone.extend(result)

    print(f"Processing {histone_name} features finished.")
    return genes_w_histone

In [9]:
# Load the E066 file
genes_df = load_gene_expression(os.path.join(DATASET_PATH, "E066.bed"))

In [10]:
genes_df

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,str,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,"""1""","""TSPAN6""",99883667,99894988,99894988
"""chrX""",99834799,99844799,"""ENSG00000000005""",0.191,1,"""0""","""TNMD""",99839799,99854882,99839799
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,"""1""","""DPM1""",49551404,49575092,49575092
"""chr1""",169858408,169868408,"""ENSG00000000457""",4.733,-1,"""1""","""SCYL3""",169818772,169863408,169863408
"""chr1""",169626245,169636245,"""ENSG00000000460""",0.942,1,"""0""","""C1orf112""",169631245,169823221,169631245
…,…,…,…,…,…,…,…,…,…,…
"""chr15""",102280913,102290913,"""ENSG00000259658""",0.212,-1,"""0""","""RP11-89K11.1""",102277302,102285913,102285913
"""chr15""",97966182,97976182,"""ENSG00000259664""",0.0,-1,"""0""","""CTD-2147F2.2""",97913601,97971182,97971182
"""chr16""",33642696,33652696,"""ENSG00000259680""",0.071,-1,"""0""","""RP11-812E19.9""",33647044,33647696,33647696


In [11]:
# Load the histone dataset
H3K9ac_df = load_histone(os.path.join(DATASET_PATH, 'E066_H3K9ac.bed'))
H3K9me3_df = load_histone(os.path.join(DATASET_PATH, 'E066_H3K9me3.bed'))
H3K4me3_df = load_histone(os.path.join(DATASET_PATH, 'E066_H3K4me3.bed'))
H3K27ac_df = load_histone(os.path.join(DATASET_PATH, 'E066_H3K27ac.bed'))
H3K27me3_df = load_histone(os.path.join(DATASET_PATH, 'E066_H3K27me3.bed'))

In [12]:
H3K9ac_df

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss,chrom_p,chromStart_p,chromEnd_p,name_p,score_p,strand_p,thickStart,thickEnd,itemRgb,blockCount,blockSizes,blockStarts,signalValue,pValue,qValue
str,i64,i64,str,f64,str,str,str,i64,i64,i64,str,i64,i64,str,f64,str,i64,i64,i64,i64,str,str,f64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,"""-1""","""1""","""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890345,99890703,"""H3K9ac_peak_58534""",13.0,""".""",99890345,99890703,0,2,"""1,1""","""0,357""",2.79934,3.26906,1.39
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,"""-1""","""1""","""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890878,99891894,"""H3K9ac_peak_58535""",52.0,""".""",99890878,99891894,0,3,"""1,853,1""","""0,19,1015""",4.4152,7.39542,5.20045
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,"""-1""","""1""","""TSPAN6""",99883667,99894988,99894988,"""chrX""",99892077,99892338,"""H3K9ac_peak_58536""",27.0,""".""",99892077,99892338,0,2,"""1,1""","""0,260""",3.47308,4.81289,2.78435
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,"""-1""","""1""","""DPM1""",49551404,49575092,49575092,"""chr20""",49573494,49576137,"""H3K9ac_peak_34601""",174.0,""".""",49573494,49576137,0,3,"""1,2506,1""","""0,134,2642""",6.61435,20.1968,17.4643
"""chr1""",169858408,169868408,"""ENSG00000000457""",4.733,"""-1""","""1""","""SCYL3""",169818772,169863408,169863408,"""chr1""",169860157,169861029,"""H3K9ac_peak_4137""",20.0,""".""",169860157,169861029,0,3,"""1,236,1""","""0,7,871""",2.78741,3.99251,2.03665
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",66869528,66879528,"""ENSG00000259471""",0.412,"""1""","""0""","""RP11-321F6.1""",66874528,66978132,66874528,"""chr15""",66874550,66874818,"""H3K9ac_peak_19254""",16.0,""".""",66874550,66874818,0,2,"""1,1""","""0,267""",2.96261,3.61327,1.69221
"""chr15""",71036271,71046271,"""ENSG00000259532""",0.0,"""1""","""0""","""RP11-138H8.2""",71041271,71046516,71041271,"""chr15""",71035484,71037250,"""H3K9ac_peak_19388""",61.0,""".""",71035484,71037250,0,4,"""1,856,661,1""","""0,37,965,1765""",4.20595,8.38599,6.14127
"""chr15""",80210113,80220113,"""ENSG00000259642""",1.753,"""1""","""0""","""C15orf37""",80215113,80217194,80215113,"""chr15""",80214890,80216796,"""H3K9ac_peak_19637""",65.0,""".""",80214890,80216796,0,5,"""1,231,869,357,1""","""0,128,413,1463,1905""",4.31376,8.82911,6.56466


In [13]:
# Build the matrix
genes_w_H3K9ac_df = get_histone_features(genes_df, H3K9ac_df, 'H3K9ac')
genes_w_H3K9me3_df = get_histone_features(genes_df, H3K9me3_df, 'H3K9me3')
genes_w_H3K4me3_df = get_histone_features(genes_df, H3K4me3_df, 'H3K4me3')
genes_w_H3K27ac_df = get_histone_features(genes_df, H3K27ac_df, 'H3K27ac')
genes_w_H3K27me3_df = get_histone_features(genes_df, H3K27me3_df, 'H3K27me3')

Processing H3K9ac: 0/19645
Processing H3K9ac: 10000/19645
Processing H3K9ac features finished.
Processing H3K9me3: 0/19645
Processing H3K9me3: 10000/19645
Processing H3K9me3 features finished.
Processing H3K4me3: 0/19645
Processing H3K4me3: 10000/19645
Processing H3K4me3 features finished.
Processing H3K27ac: 0/19645
Processing H3K27ac: 10000/19645
Processing H3K27ac features finished.
Processing H3K27me3: 0/19645
Processing H3K27me3: 10000/19645
Processing H3K27me3 features finished.


# Checking the generated features

In [14]:
genes_w_H3K9ac_df.filter(pl.col('H3K9ac_wc') > 0).sort(['H3K9ac_wc'], descending=True)

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len
str,list[f64],u32,u32
"""ENSG00000139318""","[6.05058, 6.05058, … 3.07625]",99,100
"""ENSG00000113916""","[8.2021, 8.2021, … 4.15551]",98,100
"""ENSG00000120129""","[7.87919, 7.87919, … 3.6377]",98,100
"""ENSG00000182095""","[7.21063, 7.21063, … 7.87317]",98,100
"""ENSG00000197249""","[8.4333, 8.4333, … 3.4584]",98,100
…,…,…,…
"""ENSG00000185633""","[2.94754, 0.0, … 0.0]",1,100
"""ENSG00000187908""","[0.0, 0.0, … 3.23689]",1,100
"""ENSG00000196154""","[3.21625, 0.0, … 0.0]",1,100


In [15]:
genes_w_H3K9ac_df.filter(pl.col('H3K9ac_len') < 100).sort(['H3K9ac_wc'], descending=True)

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len
str,list[f64],u32,u32


In [16]:
genes_w_H3K9me3_df.filter(pl.col('H3K9me3_wc') > 0).sort(['H3K9me3_wc'], descending=True)

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32
"""ENSG00000221843""","[3.33492, 3.33492, … 3.34726]",98,100
"""ENSG00000161551""","[4.68307, 4.68307, … 5.73526]",97,100
"""ENSG00000198046""","[4.21635, 4.21635, … 4.0086]",97,100
"""ENSG00000204920""","[4.65691, 4.65691, … 0.0]",97,100
"""ENSG00000253910""","[4.42914, 4.42914, … 4.61293]",97,100
…,…,…,…
"""ENSG00000187258""","[0.0, 0.0, … 2.86636]",1,100
"""ENSG00000198739""","[2.70339, 0.0, … 0.0]",1,100
"""ENSG00000213416""","[0.0, 0.0, … 2.74191]",1,100


In [17]:
genes_w_H3K9me3_df.filter(pl.col('H3K9me3_len') < 100).sort(['H3K9me3_wc'], descending=True)

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32


In [18]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_wc') > 0).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32
"""ENSG00000182095""","[8.63241, 8.63241, … 10.5614]",99,100
"""ENSG00000103449""","[3.3769, 3.3769, … 5.72733]",96,100
"""ENSG00000257242""","[2.79941, 2.79941, … 0.0]",92,100
"""ENSG00000139725""","[6.90556, 6.90556, … 0.0]",90,100
"""ENSG00000204920""","[3.94507, 3.94507, … 0.0]",90,100
…,…,…,…
"""ENSG00000181781""","[0.0, 0.0, … 3.31939]",1,100
"""ENSG00000188676""","[0.0, 0.0, … 3.89803]",1,100
"""ENSG00000196167""","[4.82332, 0.0, … 0.0]",1,100


In [19]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_wc') > 0).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32
"""ENSG00000182095""","[8.63241, 8.63241, … 10.5614]",99,100
"""ENSG00000103449""","[3.3769, 3.3769, … 5.72733]",96,100
"""ENSG00000257242""","[2.79941, 2.79941, … 0.0]",92,100
"""ENSG00000139725""","[6.90556, 6.90556, … 0.0]",90,100
"""ENSG00000204920""","[3.94507, 3.94507, … 0.0]",90,100
…,…,…,…
"""ENSG00000181781""","[0.0, 0.0, … 3.31939]",1,100
"""ENSG00000188676""","[0.0, 0.0, … 3.89803]",1,100
"""ENSG00000196167""","[4.82332, 0.0, … 0.0]",1,100


In [20]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_len') < 100).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32


In [21]:
genes_w_H3K27ac_df.filter(pl.col('H3K27ac_wc') > 0).sort(['H3K27ac_wc'], descending=True)

gene_id,H3K27ac,H3K27ac_wc,H3K27ac_len
str,list[f64],u32,u32
"""ENSG00000047457""","[9.95774, 9.95774, … 9.95774]",100,100
"""ENSG00000096060""","[10.0715, 10.0715, … 10.0715]",100,100
"""ENSG00000099999""","[8.16115, 8.16115, … 8.16115]",100,100
"""ENSG00000101040""","[14.0687, 14.0687, … 14.0687]",100,100
"""ENSG00000102743""","[9.40807, 9.40807, … 9.40807]",100,100
…,…,…,…
"""ENSG00000198673""","[3.4618, 0.0, … 0.0]",1,100
"""ENSG00000234776""","[0.0, 0.0, … 3.05718]",1,100
"""ENSG00000258474""","[0.0, 0.0, … 2.51105]",1,100


In [22]:
genes_w_H3K27ac_df.filter(pl.col('H3K27ac_len') < 100).sort(['H3K27ac_wc'], descending=True)

gene_id,H3K27ac,H3K27ac_wc,H3K27ac_len
str,list[f64],u32,u32


In [23]:
genes_w_H3K27me3_df.filter(pl.col('H3K27me3_wc') > 0).sort(['H3K27me3_wc'], descending=True)

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32
"""ENSG00000081087""","[3.96021, 3.96021, … 3.96021]",100,100
"""ENSG00000112333""","[3.96021, 3.96021, … 3.96021]",100,100
"""ENSG00000144290""","[3.61885, 3.61885, … 3.61885]",100,100
"""ENSG00000170178""","[3.96804, 3.96804, … 3.96804]",100,100
"""ENSG00000163081""","[3.68541, 3.68541, … 3.8533]",99,100
…,…,…,…
"""ENSG00000186051""","[2.77353, 0.0, … 0.0]",1,100
"""ENSG00000189298""","[2.78303, 0.0, … 0.0]",1,100
"""ENSG00000225899""","[2.8369, 0.0, … 0.0]",1,100


In [24]:
genes_w_H3K27me3_df.filter(pl.col('H3K27me3_len') < 100).sort(['H3K27me3_wc'], descending=True)

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32


# Join all histones into single dataframe

In [25]:
# Join all histones into single dataframe
genes_histone_df = genes_w_H3K9ac_df \
                    .join(genes_w_H3K9me3_df, on='gene_id') \
                    .join(genes_w_H3K4me3_df, on='gene_id') \
                    .join(genes_w_H3K27ac_df, on='gene_id') \
                    .join(genes_w_H3K27me3_df, on='gene_id')

In [26]:
genes_histone_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",21,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 3.33263, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",22,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",4,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",31,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",45,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",4,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",6,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100


# Join genes with the value and histone features

In [27]:
# Join genes with the value and histone features
genes_values_df = genes_df.select(['gene_id', 'E066'])
genes_values_df

gene_id,E066
str,f64
"""ENSG00000000003""",73.205
"""ENSG00000000005""",0.191
"""ENSG00000000419""",52.609
"""ENSG00000000457""",4.733
"""ENSG00000000460""",0.942
…,…
"""ENSG00000259658""",0.212
"""ENSG00000259664""",0.0
"""ENSG00000259680""",0.071


In [28]:
genes_histone_values_df = genes_histone_df.join(
    genes_values_df,
    on = 'gene_id'
)

In [29]:
genes_histone_values_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,E066
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",21,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 3.33263, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",22,100,"[0.0, 0.0, … 0.0]",0,100,73.205
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.191
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",4,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",31,100,"[0.0, 0.0, … 0.0]",0,100,52.609
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",45,100,"[0.0, 0.0, … 0.0]",0,100,4.733
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",4,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.942
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",6,100,0.0
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071


In [30]:
# Save to parquet
genes_histone_values_df.write_parquet(os.path.join(DATASET_PATH, 'E066_exp_histones.parquet'))

In [31]:
test_df = pl.read_parquet(os.path.join(DATASET_PATH, 'E066_exp_histones.parquet'))
test_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,E066
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",21,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 3.33263, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",22,100,"[0.0, 0.0, … 0.0]",0,100,73.205
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.191
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",4,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",31,100,"[0.0, 0.0, … 0.0]",0,100,52.609
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",45,100,"[0.0, 0.0, … 0.0]",0,100,4.733
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",4,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.942
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",6,100,0.0
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071


In [32]:
marker_df = pl.read_parquet(os.path.join(DATASET_PATH, "marker_combinations.parquet"))

In [33]:
marker_df

combination
list[str]
"[""H3K4me3""]"
"[""H3K9ac""]"
"[""H3K9me3""]"
"[""H3K27ac""]"
"[""H3K27me3""]"
…
"[""H3K4me3"", ""H3K9ac"", … ""H3K27me3""]"
"[""H3K4me3"", ""H3K9ac"", … ""H3K27me3""]"
"[""H3K4me3"", ""H3K9me3"", … ""H3K27me3""]"


# Prepare the index for training, validation, and testing

In [ ]:
# Convert to numpy array
all_features_np = genes_histone_values_df.to_numpy()
print(all_features_np)
print(all_features_np.shape)

In [ ]:
# Split train, test, validation by index
data_indices = np.arange(len(all_features_np))
print(data_indices)

In [ ]:
# First split: 80% train, 20% temporary (for test + validation)
train_idx, temp_idx = train_test_split(
    data_indices, 
    test_size=0.2, 
    random_state=42  # For reproducibility
)

In [ ]:
# Second split: Split temp_idx into 50% test and 50% validation
val_idx, test_idx = train_test_split(
    temp_idx, 
    test_size=0.5, 
    random_state=42  # Same random_state for consistency
)

In [ ]:
print(train_idx.shape)
print(val_idx.shape)
print(test_idx.shape)

In [ ]:
# Save train, val, and test into parquet file
train_idx_df = pd.DataFrame(train_idx, columns=['values'])
train_idx_df.to_parquet(os.path.join(DATASET_PATH, 'train_idx.parquet'))

val_idx_df = pd.DataFrame(val_idx, columns=['values'])
val_idx_df.to_parquet(os.path.join(DATASET_PATH, 'val_idx.parquet'))

test_idx_df = pd.DataFrame(test_idx, columns=['values'])
val_idx_df.to_parquet(os.path.join(DATASET_PATH, 'test_idx.parquet'))